In [ ]:
import sys
sys.path.insert(0, 'src')
from instance_generator import InstanceGenerator
from farmers_intermediaries import Instance
import pandas as pd
import os
import numpy as np
import random
from pprint import pprint
import json

In [63]:
ints_df = pd.read_csv('data/ints.csv')
farmers_df = pd.read_csv('data/farmers.csv')
instance_generator = InstanceGenerator(farmers_df, ints_df)

/Users/nachatjatusripitak/Desktop/Paper Revision/SyntheticInstanceGenerator/src/instance_generator.py:88: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


In [ ]:
N_INTS = 

In [ ]:
# =========================
# CORE BUILDERS
# =========================
def build_instance(instance_id):
    """
    Generate a fresh instance and attach the road graph.
    """
    instance_generator.gen_ints(N_INTS)

    instance_dict = instance_generator.gen_instance(
        instance_id,
        write=False,
        plot=False,
    )

    platform = Instance.from_dict(instance_dict)
    platform.set_graph(RoadGraph(GRAPH))

    return platform, instance_dict


def apply_quantity_perturbation(instance_dict, platform):
    """
    Rebuild platform with perturbed quantities.
    """
    quantities = reset_quantities(platform)
    return Instance.from_dict(instance_dict, opt_quantities=quantities)


def run_single_simulation(instance_dict, platform):
    """
    Runs one optimization on a (possibly perturbed) platform.
    """
    # Apply stochastic elements
    platform = apply_quantity_perturbation(instance_dict, platform)

    platform.set_graph(RoadGraph(GRAPH))

    epsilon = sample_epsilon(platform)
    het_costs = reset_fixed_costs(platform)

    parameters = {
        "epsilon": epsilon,
        "solver": "gurobi",
        "het_costs": het_costs,
    }

    optimizer = Optimizer(platform, parameters)

    summary = optimizer.solve(
        "heuristic_optimized",
        options={
            "structured_farmer_prices": False,
            "domination": False,
        },
    )

    # Collect outputs
    return {
        "epsilon": epsilon,
        "cost": het_costs,
        "farmer_quantities": {f.id: f.quantity for f in platform.farmers},
        "summary_vanilla": summary.to_dict(),
        "farmer_dirt_to_mill": {f.id: f.dirt_to_mill for f in platform.farmers},
        "farmer_paved_to_mill": {f.id: f.paved_to_mill for f in platform.farmers},
    }


# =========================
# JSON SERIALIZATION
# =========================
def convert(obj):
    if isinstance(obj, (np.float64, np.int64)):
        return float(obj)
    elif isinstance(obj, np.str_):
        return str(obj)
    elif isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Type {type(obj)} not serializable")


# =========================
# MAIN EXPERIMENT LOOP
# =========================
def main():
    results = []

    for sim_n in range(SIM_SIZE):
        print(f"--- Simulation {sim_n + 1}/{SIM_SIZE} ---")

        instance_id = f"{n_id}_{sim_n}"

        # Build new instance (between-instance variation)
        platform, instance_dict = build_instance(instance_id)

        # Run one stochastic solve
        sim_result = run_single_simulation(instance_dict, platform)

        # Add metadata
        sim_result.update({
            "instance_id": instance_id,
            "n_id": n_id,
            "sim_n": sim_n,
        })

        results.append(sim_result)

        print(f"Completed simulation {sim_n}")
        print()

    # Save results
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=4, default=convert)

    print(f"Results saved to {RESULTS_PATH}")


dict_keys(['Competent Mayer', 'Wizardly Satoshi', 'Epic Brattain', 'Unruffled Joliot', 'Romantic Heisenberg', 'Eager Booth', 'Agitated Williamson', 'Hopeful Jang', 'Unruffled Shamir', 'Stoic Johnson', 'Suspicious Spence', 'Pensive Bose', 'Agitated Archimedes', 'Romantic Liskov', 'Heuristic Payne', 'Intelligent Montalcini', 'Nervous Diffie', 'Jovial Kirch', 'Elated Goodall', 'Sharp Diffie', 'Kind Fermi', 'Interesting Tharp', 'Reverent Borg', 'Recursing Montalcini', 'Focused Brattain', 'Gifted Wilson', 'Stupefied Kowalevski', 'Modest Tereshkova', 'Distracted Gagarin', 'Pensive Wilbur', 'Modest Hertz', 'Amazing Kepler', 'Thirsty Solomon', 'Interesting Noyce', 'Happy Lumiere', 'Vibrant Pare', 'Hardcore Blackwell', 'Sad Lamarr', 'Vibrant Chatelet', 'Zen Hertz'])
{'farmers': [{'id': 'Competent Mayer_f0',
              'intermediary': 'Competent Mayer',
              'location': [-0.3673985319733595, 102.44321250500504],
              'quantity': 1.0},
             {'id': 'Competent Mayer_f1'